### Import


In [1]:
from pyspark.sql import SparkSession

### Create SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("Retail Iceberg")
    .config(
        "spark.sql.catalog.retail",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        "spark.sql.catalog.retail.type",
        "hadoop"
    )
    .config(
        "spark.sql.catalog.retail.warehouse",
        "/home/iceberg/warehouse"
    )
    .config(
        "spark.sql.defaultCatalog",
        "retail"
    )
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/25 05:17:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/25 05:17:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
spark 


In [4]:
print(spark.version)

3.5.5


### Testing id the Iceberg catalog exists

In [5]:
spark.sql("show catalogs").show()

+-------------+
|      catalog|
+-------------+
|       retail|
|spark_catalog|
+-------------+



In [6]:
spark.conf.get("spark.sql.catalog.retail", "NOT SET")

'org.apache.iceberg.spark.SparkCatalog'

In [7]:
spark.conf.get("spark.sql.catalog.retail.type", "NOT SET")

'hadoop'

In [8]:
spark.conf.get("spark.sql.catalog.retail.uri", "NOT SET")

'NOT SET'

In [9]:
spark.conf.get("spark.sql.defaultCatalog", "NOT SET")

'retail'

In [10]:
spark.conf.get("spark.sql.catalog.demo")

'org.apache.iceberg.spark.SparkCatalog'

In [11]:
spark.conf.get("spark.sql.catalog.demo.uri")

'http://rest:8181'

### Creating the first DB

In [12]:
spark.sql("CREATE DATABASE IF NOT EXISTS retail.sales_db") #retail=catalog and sales_db=database

DataFrame[]

In [3]:
spark.sql("SHOW DATABASES IN retail").show()

+---------+
|namespace|
+---------+
| sales_db|
+---------+



### A table

In [23]:
spark.sql("""
    CREATE TABLE retail.sales_db.sales(
        transaction_id BIGINT,
        product STRING,
        category STRING,
        quantity INT,
        price DOUBLE,
        sales_date DATE
    )
    USING iceberg
"""
)

DataFrame[]

In [14]:
spark.sql("SHOW TABLES IN retail.sales_db").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
| sales_db|    sales|      false|
+---------+---------+-----------+



### Inserting data

In [26]:
spark.sql("""
    INSERT INTO retail.sales_db.sales VALUES
    (1,'Laptop','Electronics',1,1200,DATE '2026-08-01'),
    (2,'Mouse','Electronics',2,25,DATE '2026-08-01'),
    (3,'Keyboard','Electronics',1,75,DATE '2026-08-02'),
    (4,'Chair','Furniture',1,180,DATE '2026-08-02'),
    (5,'Desk','Furniture',1,350,DATE '2026-08-03')
""")

DataFrame[]

In [15]:
spark.sql("SELECT * FROM retail.sales_db.sales").show()

+--------------+----------+-----------+--------+------+----------+
|transaction_id|   product|   category|quantity| price|sales_date|
+--------------+----------+-----------+--------+------+----------+
|             1|    Laptop|Electronics|       1|1200.0|2026-08-01|
|             6|   Monitor|Electronics|       1| 300.0|2026-08-04|
|             1|    Laptop|Electronics|       1|1200.0|2026-08-01|
|             6|   Monitor|Electronics|       1| 300.0|2026-08-04|
|             2|     Mouse|Electronics|       2|  25.0|2026-08-01|
|             2|     Mouse|Electronics|       2|  25.0|2026-08-01|
|             7|Headphones|Electronics|       1| 150.0|2026-08-04|
|             3|  Keyboard|Electronics|       1|  75.0|2026-08-02|
|             7|Headphones|Electronics|       1| 150.0|2026-08-04|
|             3|  Keyboard|Electronics|       1|  75.0|2026-08-02|
|             4|     Chair|  Furniture|       1| 180.0|2026-08-02|
|             8|     Table|  Furniture|       1| 400.0|2026-08

### Time Table

#### Create new snapshot

In [28]:
spark.sql("""
    INSERT INTO retail.sales_db.sales VALUES
    (6,'Monitor','Electronics',1,300,DATE '2026-08-04'),
    (7,'Headphones','Electronics',1,150,DATE '2026-08-04'),
    (8,'Table','Furniture',1,400,DATE '2026-08-04')
""")

DataFrame[]

In [16]:
spark.sql("""
    SELECT * 
    FROM retail.sales_db.sales
    ORDER BY transaction_id
""").show()

+--------------+----------+-----------+--------+------+----------+
|transaction_id|   product|   category|quantity| price|sales_date|
+--------------+----------+-----------+--------+------+----------+
|             1|    Laptop|Electronics|       1|1200.0|2026-08-01|
|             1|    Laptop|Electronics|       1|1200.0|2026-08-01|
|             2|     Mouse|Electronics|       2|  25.0|2026-08-01|
|             2|     Mouse|Electronics|       2|  25.0|2026-08-01|
|             3|  Keyboard|Electronics|       1|  75.0|2026-08-02|
|             3|  Keyboard|Electronics|       1|  75.0|2026-08-02|
|             4|     Chair|  Furniture|       1| 180.0|2026-08-02|
|             4|     Chair|  Furniture|       1| 180.0|2026-08-02|
|             5|      Desk|  Furniture|       1| 350.0|2026-08-03|
|             5|      Desk|  Furniture|       1| 350.0|2026-08-03|
|             6|   Monitor|Electronics|       1| 300.0|2026-08-04|
|             6|   Monitor|Electronics|       1| 300.0|2026-08

### Checking snapshots

In [17]:
spark.sql("""
SELECT *
FROM retail.sales_db.sales
ORDER BY transaction_id
""").show()

+--------------+----------+-----------+--------+------+----------+
|transaction_id|   product|   category|quantity| price|sales_date|
+--------------+----------+-----------+--------+------+----------+
|             1|    Laptop|Electronics|       1|1200.0|2026-08-01|
|             1|    Laptop|Electronics|       1|1200.0|2026-08-01|
|             2|     Mouse|Electronics|       2|  25.0|2026-08-01|
|             2|     Mouse|Electronics|       2|  25.0|2026-08-01|
|             3|  Keyboard|Electronics|       1|  75.0|2026-08-02|
|             3|  Keyboard|Electronics|       1|  75.0|2026-08-02|
|             4|     Chair|  Furniture|       1| 180.0|2026-08-02|
|             4|     Chair|  Furniture|       1| 180.0|2026-08-02|
|             5|      Desk|  Furniture|       1| 350.0|2026-08-03|
|             5|      Desk|  Furniture|       1| 350.0|2026-08-03|
|             6|   Monitor|Electronics|       1| 300.0|2026-08-04|
|             6|   Monitor|Electronics|       1| 300.0|2026-08

In [18]:
spark.sql("""
    SELECT *
    FROM retail.sales_db.sales.snapshots
""").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                       |summary                                                                                                                                                   

In [19]:
snapshots = spark.sql("""
    SELECT snapshot_id,parent_id,committed_At,operation
    FROM retail.sales_db.sales.snapshots
    ORDER BY committed_at

""")

In [20]:
snapshots.show(truncate=False)

+-------------------+-------------------+-----------------------+---------+
|snapshot_id        |parent_id          |committed_At           |operation|
+-------------------+-------------------+-----------------------+---------+
|1350242478421060038|NULL               |2026-08-06 09:22:33.619|append   |
|713981930178453912 |1350242478421060038|2026-08-06 15:56:10.307|append   |
|1754596836953751592|713981930178453912 |2026-08-24 05:18:51.638|append   |
|2160569991645849900|1754596836953751592|2026-08-24 05:18:52.752|append   |
+-------------------+-------------------+-----------------------+---------+



### Using Time Travel

In [21]:
spark.sql("""
    SELECT *
    FROM retail.sales_db.sales
    VERSION AS OF 1350242478421060038
    ORDER BY transaction_id
""").show()

+--------------+--------+-----------+--------+------+----------+
|transaction_id| product|   category|quantity| price|sales_date|
+--------------+--------+-----------+--------+------+----------+
|             1|  Laptop|Electronics|       1|1200.0|2026-08-01|
|             2|   Mouse|Electronics|       2|  25.0|2026-08-01|
|             3|Keyboard|Electronics|       1|  75.0|2026-08-02|
|             4|   Chair|  Furniture|       1| 180.0|2026-08-02|
|             5|    Desk|  Furniture|       1| 350.0|2026-08-03|
+--------------+--------+-----------+--------+------+----------+



### Timetravel using committed time

In [22]:
spark.sql("""
    SELECT *
    FROM retail.sales_db.sales
    TIMESTAMP AS OF '2026-08-06 09:22:33.619'
    ORDER BY transaction_id
""").show()

+--------------+--------+-----------+--------+------+----------+
|transaction_id| product|   category|quantity| price|sales_date|
+--------------+--------+-----------+--------+------+----------+
|             1|  Laptop|Electronics|       1|1200.0|2026-08-01|
|             2|   Mouse|Electronics|       2|  25.0|2026-08-01|
|             3|Keyboard|Electronics|       1|  75.0|2026-08-02|
|             4|   Chair|  Furniture|       1| 180.0|2026-08-02|
|             5|    Desk|  Furniture|       1| 350.0|2026-08-03|
+--------------+--------+-----------+--------+------+----------+



### Schema Evolution

In [ ]:
spark.sql("""
    DESCRIBE retail.sales_db.sales
""").show(truncate=False)

+--------------+---------+-------+
|col_name      |data_type|comment|
+--------------+---------+-------+
|transaction_id|bigint   |NULL   |
|product       |string   |NULL   |
|category      |string   |NULL   |
|quantity      |int      |NULL   |
|price         |double   |NULL   |
|sales_date    |date     |NULL   |
+--------------+---------+-------+



#### Adding the new column

In [5]:
spark.sql("""
    ALTER TABLE retail.sales_db.sales
    ADD COLUMN store_city STRING
""")

DataFrame[]

In [6]:
spark.sql("""
    DESCRIBE retail.sales_db.sales
""").show(truncate=False)

+--------------+---------+-------+
|col_name      |data_type|comment|
+--------------+---------+-------+
|transaction_id|bigint   |NULL   |
|product       |string   |NULL   |
|category      |string   |NULL   |
|quantity      |int      |NULL   |
|price         |double   |NULL   |
|sales_date    |date     |NULL   |
|store_city    |string   |NULL   |
+--------------+---------+-------+



#### The new column is created but the iceberg schema tells spark that older data doesn't have a value, so we will get null

In [9]:
spark.sql("""
    SELECT *
    FROM retail.sales_db.sales
    ORDER BY transaction_id
""").show()

+--------------+----------+-----------+--------+------+----------+----------+
|transaction_id|   product|   category|quantity| price|sales_date|store_city|
+--------------+----------+-----------+--------+------+----------+----------+
|             1|    Laptop|Electronics|       1|1200.0|2026-08-01|      NULL|
|             1|    Laptop|Electronics|       1|1200.0|2026-08-01|      NULL|
|             2|     Mouse|Electronics|       2|  25.0|2026-08-01|      NULL|
|             2|     Mouse|Electronics|       2|  25.0|2026-08-01|      NULL|
|             3|  Keyboard|Electronics|       1|  75.0|2026-08-02|      NULL|
|             3|  Keyboard|Electronics|       1|  75.0|2026-08-02|      NULL|
|             4|     Chair|  Furniture|       1| 180.0|2026-08-02|      NULL|
|             4|     Chair|  Furniture|       1| 180.0|2026-08-02|      NULL|
|             5|      Desk|  Furniture|       1| 350.0|2026-08-03|      NULL|
|             5|      Desk|  Furniture|       1| 350.0|2026-08-0

#### Checking for the snapshot

In [10]:
spark.sql("""
    SELECT snapshot_id, parent_id, committed_at, operation
    FROM retail.sales_db.sales.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-------------------+-------------------+-----------------------+---------+
|snapshot_id        |parent_id          |committed_at           |operation|
+-------------------+-------------------+-----------------------+---------+
|1350242478421060038|NULL               |2026-08-06 09:22:33.619|append   |
|713981930178453912 |1350242478421060038|2026-08-06 15:56:10.307|append   |
|1754596836953751592|713981930178453912 |2026-08-24 05:18:51.638|append   |
|2160569991645849900|1754596836953751592|2026-08-24 05:18:52.752|append   |
+-------------------+-------------------+-----------------------+---------+



In [11]:
spark.sql("""
    SELECT *
    FROM retail.sales_db.sales.history
    ORDER BY made_current_at
""").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-08-06 09:22:33.619|1350242478421060038|NULL               |true               |
|2026-08-06 15:56:10.307|713981930178453912 |1350242478421060038|true               |
|2026-08-24 05:18:51.638|1754596836953751592|713981930178453912 |true               |
|2026-08-24 05:18:52.752|2160569991645849900|1754596836953751592|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [12]:
spark.sql("""
    SELECT *
    FROM retail.sales_db.sales.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                       |summary                                                                                                                                                   

#### Insering new data containind store_city

In [13]:
spark.sql("""
    INSERT INTO retail.sales_db.sales VALUES
    (9, 'Laptop Stand', 'Accessories', 1, 45.0, DATE '2026-08-06', 'Kathmandu'),
    (10, 'Headphones', 'Electronics', 2, 80.0, DATE '2026-08-06', 'Pokhara'),
    (11, 'Office Chair', 'Furniture', 1, 220.0, DATE '2026-08-07', 'Lalitpur')
""")

DataFrame[]

In [14]:
spark.sql("""
    SELECT *
    FROM retail.sales_db.sales
    ORDER BY transaction_id
""").show(truncate=False)

+--------------+------------+-----------+--------+------+----------+----------+
|transaction_id|product     |category   |quantity|price |sales_date|store_city|
+--------------+------------+-----------+--------+------+----------+----------+
|1             |Laptop      |Electronics|1       |1200.0|2026-08-01|NULL      |
|1             |Laptop      |Electronics|1       |1200.0|2026-08-01|NULL      |
|2             |Mouse       |Electronics|2       |25.0  |2026-08-01|NULL      |
|2             |Mouse       |Electronics|2       |25.0  |2026-08-01|NULL      |
|3             |Keyboard    |Electronics|1       |75.0  |2026-08-02|NULL      |
|3             |Keyboard    |Electronics|1       |75.0  |2026-08-02|NULL      |
|4             |Chair       |Furniture  |1       |180.0 |2026-08-02|NULL      |
|4             |Chair       |Furniture  |1       |180.0 |2026-08-02|NULL      |
|5             |Desk        |Furniture  |1       |350.0 |2026-08-03|NULL      |
|5             |Desk        |Furniture  

In [16]:
spark.sql("""
    SELECT snapshot_id, parent_id, committed_at, operation
    FROM retail.sales_db.sales.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-------------------+-------------------+-----------------------+---------+
|snapshot_id        |parent_id          |committed_at           |operation|
+-------------------+-------------------+-----------------------+---------+
|1350242478421060038|NULL               |2026-08-06 09:22:33.619|append   |
|713981930178453912 |1350242478421060038|2026-08-06 15:56:10.307|append   |
|1754596836953751592|713981930178453912 |2026-08-24 05:18:51.638|append   |
|2160569991645849900|1754596836953751592|2026-08-24 05:18:52.752|append   |
|6349726583116857765|2160569991645849900|2026-08-25 05:34:33.302|append   |
+-------------------+-------------------+-----------------------+---------+



In [20]:
spark.sql("""
    SELECT transaction_id, product, store_city
    FROM retail.sales_db.sales
    ORDER BY transaction_id
""").show()
# Iceberg doesn't need to rewrite the old files when a new column is added, the new column will be null for the old records.

+--------------+------------+----------+
|transaction_id|     product|store_city|
+--------------+------------+----------+
|             1|      Laptop|      NULL|
|             1|      Laptop|      NULL|
|             2|       Mouse|      NULL|
|             2|       Mouse|      NULL|
|             3|    Keyboard|      NULL|
|             3|    Keyboard|      NULL|
|             4|       Chair|      NULL|
|             4|       Chair|      NULL|
|             5|        Desk|      NULL|
|             5|        Desk|      NULL|
|             6|     Monitor|      NULL|
|             6|     Monitor|      NULL|
|             7|  Headphones|      NULL|
|             7|  Headphones|      NULL|
|             8|       Table|      NULL|
|             8|       Table|      NULL|
|             9|Laptop Stand| Kathmandu|
|            10|  Headphones|   Pokhara|
|            11|Office Chair|  Lalitpur|
+--------------+------------+----------+



### Using TimeTravel again

In [22]:
spark.sql("""
    SELECT *
    FROM retail.sales_db.sales
    VERSION AS OF 1350242478421060038
    ORDER BY transaction_id
""").show()
# We get the old record without the new column store_city, because the record was created before the column was added.

+--------------+--------+-----------+--------+------+----------+
|transaction_id| product|   category|quantity| price|sales_date|
+--------------+--------+-----------+--------+------+----------+
|             1|  Laptop|Electronics|       1|1200.0|2026-08-01|
|             2|   Mouse|Electronics|       2|  25.0|2026-08-01|
|             3|Keyboard|Electronics|       1|  75.0|2026-08-02|
|             4|   Chair|  Furniture|       1| 180.0|2026-08-02|
|             5|    Desk|  Furniture|       1| 350.0|2026-08-03|
+--------------+--------+-----------+--------+------+----------+

